# Problem Setup

**Laplace Operator($\Delta$)**

$$\Delta u=u_{xx}+u_{yy} = \frac{\partial^2 u}{\partial x^2}+\frac{\partial^2 u}{\partial y^2}$$

**Harmonic Functions**

$$\Delta u =0$$

**Poisson equation**

$$\Delta u =f(x)$$

Example: Candle in a room

### Problem

$$-\Delta u=\pi^2sin(\pi x)$$

### Boundary Conditions:

$$u(-1)=0, u(1) =0$$

### Exact solution:

$$u(x)=sin(\pi x)$$

### PDE:
$$-\Delta u=\pi^2sin(\pi x)$$
$$u_{xx}=-\pi^2sin(\pi x)$$
$$\frac{\partial^2u}{\partial x^2}=-\pi^2sin(\pi x)$$

So the residual will be:

$$0=\frac{\partial^2u}{\partial x^2}+\pi^2sin(\pi x)$$

Note: Remeber that our neural network $NN(x)\approx u(x)$ so:

$$\frac{\partial^2NN}{\partial x^2}\approx\frac{\partial^2u}{\partial x^2}$$

$$f=\frac{\partial^2NN}{\partial x^2}+\pi^2sin(\pi x)\rightarrow 0$$


**Boundary Conditions**
$$u(-1)=0$$

$$u(1)=0$$

# Libraries

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import torch
from torch.autograd import grad
from pyDOE import lhs

from Lib.ImportLibrary import configure_notebook
from Lib.MultiLayerNet import MultiLayerNet

dev = configure_notebook()
output_dir = Path("output/Case3")
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint = output_dir / "model.pt"

# Training parameters

In [ ]:
steps=5000
lr=1e-3
layers = [1,30,30,30,30,1]
# To generate new data:
x_min=-1
x_max=1
total_points=500
#Nu: Number of training points (2 as we onlt have 2 boundaries), # Nf: Number of collocation points (Evaluate PDE)
Nu=2
Nf=250

## Functions

In [ ]:
def f_BC(x):
  return 1-torch.abs(x)
def f_real(x):
  return torch.sin(torch.pi*x)
def PDE(x):
  return -1*(torch.pi**2)*torch.sin(torch.pi*x)


In [ ]:
def mygrad(y, x):
    dydx = grad(y, x, torch.ones_like(y),
                create_graph=True, retain_graph=True)[0]
    return dydx

# Generate data

In [ ]:
# get the analytical solution over the full domain
x = torch.linspace(x_min,x_max,total_points).view(-1,1) #prepare to NN
y = f_real(x)
print(x.shape, y.shape)


In [ ]:
fig, ax1 = plt.subplots()
ax1.plot(x.detach().numpy(),y.detach().numpy(),color='blue',label='Real Function')
#ax1.plot(x_train.detach().numpy(),yh.detach().numpy(),color='red',label='Pred_Train')
ax1.set_xlabel('x',color='black')
ax1.set_ylabel('f(x)',color='black')
ax1.tick_params(axis='y', color='black')
ax1.legend(loc = 'upper left')

In [ ]:
BC_1=x[0,:]
BC_2=x[-1,:]
# Total Tpaining points BC1+BC2
x_BC=torch.vstack([BC_1,BC_2])
#Select Nf points
# Latin Hypercube sampling for collocation points 
x_PDE = BC_1 + (BC_2-BC_1)*torch.as_tensor(lhs(1, Nf), dtype=x.dtype)
x_PDE = torch.vstack((x_PDE,x_BC)) 

# Train Neural Network

In [ ]:
#Store tensors to GPU
x_PDE=x_PDE.float().to(dev)
x_BC=x_BC.to(dev)
x_PDE.requires_grad_(True)
x_BC.requires_grad_(True)
y_BC=f_BC(x_BC).to(dev)

#Create Model
model = MultiLayerNet(layers, init=True)
model.to(dev)
optimizer = torch.optim.Adam(model.parameters(),lr=lr,amsgrad=False)
start_time = time.time()

In [ ]:
start_time = time.time()
it_time = time.time()
for i in range(steps):
    u = model(x_PDE)
    BCu = model(x_BC)
    dudx = mygrad(u, x_PDE)
    dudx2 = mygrad(dudx, x_PDE)
    loss_PDE = torch.mean(torch.square(dudx2 + torch.pi ** 2 * torch.sin(torch.pi * x_PDE)))
    loss_BC = torch.mean(torch.square(BCu))
    loss = loss_PDE+loss_BC
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if i % max(1, steps // 10) == 0:
      print('Iter: %d  Loss: %.5e  Loss_PDE: %.5e  Loss_BC: %.5e  Time: %.3e'
                    % (i, loss.item(), loss_PDE.item(), loss_BC.item(), time.time() - it_time))
      it_time = time.time()

torch.save(model.state_dict(), checkpoint)

# Test Model

In [ ]:
# Function
yh=model(x.to(dev))
y=f_real(x)

In [ ]:
# Derivative
g=x.to(dev)
g=g.clone()
g.requires_grad=True
f=model(g)
f_x=mygrad(f,g)

In [ ]:
# Detach from GPU
y_plot=y.detach().numpy()
yh_plot=yh.detach().cpu().numpy()
f_x_plot=f_x.detach().cpu().numpy()

In [ ]:
# Plot
fig, ax1 = plt.subplots()
ax1.plot(x,y_plot,'--',color='blue',label='Real u(x)')
ax1.plot(x,yh_plot,':',color='red',label='Predicted u(x)')
ax1.plot(x,f_x_plot,color='green',label='Computed u\'(x)')
ax1.set_xlabel('x',color='black')
ax1.set_ylabel('f(x)',color='black')
ax1.tick_params(axis='y', color='black')
ax1.legend(loc = 'upper left')